# Baseline (TensorFlow / Keras): ResNet50 on 5 Galaxy Zoo 2 classes

ImageNet-pretrained ResNet50, fine-tuned end to end at 128 px with bf16 mixed precision.

### 1. Setup: GPU, libraries and run settings
Loads the CUDA libraries TensorFlow needs (otherwise it silently falls back to the CPU), imports everything, and defines the run settings in `UPPER_CASE`
(epochs, batch size, peak learning rate, seed, output folder). It seeds the random number generators, turns on bf16 mixed precision,
and stops with an error if no GPU is visible.

In [ ]:
# TensorFlow's pip CUDA libraries live in site-packages/nvidia/*/lib and are not on the loader path in this
# environment, so TF silently falls back to CPU. Preload them (must happen BEFORE importing tensorflow).
import ctypes, glob, os, site
nvidia_lib_root = os.path.join(site.getsitepackages()[0], 'nvidia')
cuda_library_dirs = ['cuda_runtime', 'cublas', 'cudnn', 'cufft', 'curand', 'cusolver', 'cusparse', 'nvjitlink', 'cuda_nvrtc', 'cuda_cupti']
for library_dir in cuda_library_dirs:
    for library_path in sorted(glob.glob(f'{nvidia_lib_root}/{library_dir}/lib/lib*.so.*')):
        try: ctypes.CDLL(library_path, mode=ctypes.RTLD_GLOBAL)
        except OSError as error: print('could not preload', library_path, error)

import json, time, numpy as np, pandas as pd, tensorflow as tf
gpu_devices = tf.config.list_physical_devices('GPU')
for gpu in gpu_devices: tf.config.experimental.set_memory_growth(gpu, True)
import keras
from keras import layers
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score

PROJECT_ROOT = Path('..').resolve() if Path('..', 'data').exists() else Path('.').resolve()
PROCESSED_DIR = PROJECT_ROOT/'data/processed'
SMOKE_TEST = os.environ.get('MV_SMOKE') == '1'
RUN_DIR = PROJECT_ROOT/('runs/smoke_tf' if SMOKE_TEST else 'runs/baseline_tf_5class'); RUN_DIR.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS, BATCH_SIZE, PEAK_LEARNING_RATE, SEED = (1 if SMOKE_TEST else 15), 128, 1e-3, 42
keras.utils.set_random_seed(SEED); random_generator = np.random.default_rng(SEED)
keras.mixed_precision.set_global_policy('mixed_bfloat16')
print('smoke' if SMOKE_TEST else 'full', '| epochs', NUM_EPOCHS, '| tf', tf.__version__, '| keras', keras.__version__, '| gpus', gpu_devices)
assert gpu_devices, 'TensorFlow does not see the GPU - it would train on CPU'

### 2. Load the labels and images
Reads `labels.csv` and the 128 px image cache into memory (about 5 GB). The held-out class (`merger`) is removed from the classifier, leaving 5 classes.
`split_indices` records which image rows belong to train, validation and test; `held_out_indices` keeps the merger rows for the later VAE test.
In smoke-test mode the train and validation sets are shrunk so the run is quick.

In [ ]:
label_table = pd.read_csv(PROCESSED_DIR/'labels.csv')
HELD_OUT_CLASSES = {'merger'}                      # classes kept out of the classifier (see top markdown)
class_names = [name for name in label_table.drop_duplicates('label').sort_values('label').cls if name not in HELD_OUT_CLASSES]
assert (label_table[~label_table.cls.isin(HELD_OUT_CLASSES)].label.max() == len(class_names) - 1)   # remaining labels are contiguous 0..K-1
labels = label_table.label.values.astype('int32')            # one class id per image row
images = np.load(PROCESSED_DIR/'images_128.npy')             # uint8, fully in RAM (~5 GB)
IMAGE_SIZE = images.shape[1]
is_in_classifier = ~label_table.cls.isin(HELD_OUT_CLASSES)
split_indices = {split: np.where((label_table.split == split) & is_in_classifier)[0] for split in ('train', 'val', 'test')}
held_out_indices = np.where(~is_in_classifier)[0]            # e.g. the 95 mergers, for the VAE anomaly test later
if SMOKE_TEST:
    split_indices['train'] = np.sort(random_generator.choice(split_indices['train'], 12000, replace=False)); split_indices['val'] = split_indices['val'][:3000]
print({split: len(rows) for split, rows in split_indices.items()}, '| held out:', len(held_out_indices), '|', class_names, images.shape)

### 3. Batch pipeline
`make_dataset` turns a list of image rows into a stream of `(images, labels)` batches for Keras. Training batches are reshuffled every epoch and the last partial batch is dropped;
validation and test batches keep their original order. The stream's length is declared up front so Keras knows how many steps make an epoch.

In [ ]:
def make_dataset(example_indices, batch_size, shuffle):
    """tf.data pipeline over the in-RAM uint8 array. Shuffled: drops the last partial batch. Otherwise keeps order."""
    num_batches = len(example_indices) // batch_size if shuffle else -(-len(example_indices) // batch_size)   # floor if shuffling, else ceil
    def generate_batches():
        ordered_indices = random_generator.permutation(example_indices) if shuffle else example_indices
        for batch_number in range(num_batches):
            batch_indices = ordered_indices[batch_number * batch_size:(batch_number + 1) * batch_size]
            yield images[batch_indices], labels[batch_indices]
    batch_signature = (tf.TensorSpec((None, IMAGE_SIZE, IMAGE_SIZE, 3), tf.uint8), tf.TensorSpec((None,), tf.int32))
    dataset = tf.data.Dataset.from_generator(generate_batches, output_signature=batch_signature).apply(tf.data.experimental.assert_cardinality(num_batches))
    return dataset.prefetch(tf.data.AUTOTUNE)   # known length -> Keras knows steps per epoch

### 4. Augmentation and preprocessing layers
Two small custom layers that live inside the model, so `predict` on validation and test skips the augmentation automatically.
`RandomOrientation` (training only) rotates each image by a random multiple of 90 degrees and randomly flips it, since galaxies have no preferred orientation.
`ImageNetPreprocess` converts raw 0-255 RGB pixels into the format the pretrained ResNet50 expects.

In [ ]:
class RandomOrientation(layers.Layer):
    """Random 90-degree rotation + random flip, independently per image (galaxies have no preferred orientation)."""
    def call(self, image_batch, training=None):
        if not training: return image_batch
        quarter_turns = tf.random.uniform([tf.shape(image_batch)[0], 1, 1, 1], 0, 4, dtype=tf.int32)
        rotated_batch = tf.where(quarter_turns == 1, tf.image.rot90(image_batch, 1),
                        tf.where(quarter_turns == 2, tf.image.rot90(image_batch, 2),
                        tf.where(quarter_turns == 3, tf.image.rot90(image_batch, 3), image_batch)))
        return tf.image.random_flip_left_right(rotated_batch)

class ImageNetPreprocess(layers.Layer):
    """uint8 [0,255] RGB -> what keras.applications.ResNet50 expects (BGR, ImageNet mean subtracted)."""
    def call(self, image_batch): return keras.applications.resnet50.preprocess_input(tf.cast(image_batch, tf.float32))

### 5. Build the model
Chains the pieces: raw uint8 images, then random orientation (training only), then ImageNet preprocessing, then ResNet50 with ImageNet weights (every layer stays trainable),
then one dense layer that outputs a score (logit) for each of the 5 classes. The final layer is kept in float32 even though the rest runs in bf16.

In [ ]:
image_input = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), dtype='uint8')
preprocessed_images = ImageNetPreprocess()(RandomOrientation()(image_input))
resnet_backbone = keras.applications.ResNet50(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), pooling='avg')
class_logits = layers.Dense(len(class_names), dtype='float32')(resnet_backbone(preprocessed_images))   # float32 logits even under mixed precision
model = keras.Model(image_input, class_logits)

### 6. Class weights
Spirals are far more common than other classes, so the loss gives rarer classes more weight. Each weight is the inverse of the class frequency, raised to the power 0.5
to soften it: that helps the rarer classes (e.g. `smooth_cigar`, 1,108 training examples) without letting spirals (48,518) dominate. The weights are printed for inspection.

In [ ]:
train_class_counts = np.bincount(labels[split_indices['train']], minlength=len(class_names))
class_weight_values = (train_class_counts.sum() / (len(class_names) * train_class_counts)) ** 0.5
class_weights = {class_id: float(weight) for class_id, weight in enumerate(class_weight_values)}
print({name: round(float(weight), 2) for name, weight in zip(class_names, class_weight_values)}, f'| params {model.count_params()/1e6:.1f}M')

### 7. Learning-rate schedule and compile
The learning rate warms up linearly for the first 20% of training up to its peak, then decays along a cosine curve toward zero. The model is compiled with the AdamW optimizer,
cross-entropy loss on the logits, and accuracy as a running metric. `steps_per_epoch` (batches per epoch) is also used later for the speed readout.

In [ ]:
steps_per_epoch = len(split_indices['train']) // BATCH_SIZE
total_steps, warmup_steps = NUM_EPOCHS * steps_per_epoch, int(0.2 * NUM_EPOCHS * steps_per_epoch)
learning_rate_schedule = keras.optimizers.schedules.CosineDecay(initial_learning_rate=PEAK_LEARNING_RATE / 25, decay_steps=total_steps - warmup_steps,
                                                                 warmup_target=PEAK_LEARNING_RATE, warmup_steps=warmup_steps)   # warmup then cosine decay
model.compile(optimizer=keras.optimizers.AdamW(learning_rate_schedule, weight_decay=1e-4),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'], jit_compile=False)

### 8. Per-epoch validation callback
After every training epoch this callback scores the model on the validation split (loss, accuracy, macro-F1), saves the weights whenever macro-F1 reaches a new best
(`best.weights.h5`), and prints and appends one summary line to `runs/baseline_tf_5class/log.txt`. The best-epoch choice therefore uses validation data only.

In [ ]:
class ValidationEvaluator(keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_macro_f1, self.epoch_records, self.log_file = -1, [], open(RUN_DIR/'log.txt', 'a')
    def on_epoch_begin(self, epoch, logs=None): self.epoch_start_time = time.time()
    def on_epoch_end(self, epoch, logs=None):
        train_seconds = time.time() - self.epoch_start_time
        val_logits = self.model.predict(make_dataset(split_indices['val'], 512, False), verbose=0)
        true_labels, predicted_labels = labels[split_indices['val']], val_logits.argmax(1)
        val_loss = float(tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(true_labels, val_logits)))   # unweighted CE
        val_macro_f1, val_accuracy = f1_score(true_labels, predicted_labels, average='macro'), float((predicted_labels == true_labels).mean())
        self.epoch_records.append(dict(epoch=epoch + 1, train_loss=logs['loss'], val_loss=val_loss, val_macro_f1=val_macro_f1, val_acc=val_accuracy, sec=train_seconds))
        if val_macro_f1 > self.best_macro_f1: self.best_macro_f1 = val_macro_f1; self.model.save_weights(RUN_DIR/'best.weights.h5')
        line = f'ep {epoch+1:2d}/{NUM_EPOCHS} | train {logs["loss"]:.3f} | val {val_loss:.3f} acc {val_accuracy:.3f} macroF1 {val_macro_f1:.3f} | {train_seconds:.0f}s ({steps_per_epoch*BATCH_SIZE/train_seconds:.0f} img/s)'
        print(line, flush=True); self.log_file.write(line + '\n'); self.log_file.flush()

### 9. Train
Runs the training loop with the class weights and the validation callback, then writes the per-epoch numbers to `history.json`.
Keras' own progress output is switched off because the callback prints one line per epoch.

In [ ]:
validation_evaluator = ValidationEvaluator()
model.fit(make_dataset(split_indices['train'], BATCH_SIZE, True), epochs=NUM_EPOCHS, class_weight=class_weights, callbacks=[validation_evaluator], shuffle=False, verbose=0)
json.dump(validation_evaluator.epoch_records, open(RUN_DIR/'history.json', 'w'), indent=1)

### 10. Training curves
Left: training loss (class-weighted, with augmentation) next to validation loss (unweighted), which are computed differently and not directly comparable.
Right: validation macro-F1 and accuracy per epoch. A large gap between the two means some classes are doing much worse than others.

In [ ]:
history_table = pd.DataFrame(validation_evaluator.epoch_records)
figure, (loss_axis, validation_axis) = plt.subplots(1, 2, figsize=(10, 3.5))
loss_axis.plot(history_table.epoch, history_table.train_loss, label='train (weighted)'); loss_axis.plot(history_table.epoch, history_table.val_loss, label='val (unweighted)'); loss_axis.set_title('loss'); loss_axis.legend()
validation_axis.plot(history_table.epoch, history_table.val_macro_f1, label='macro F1'); validation_axis.plot(history_table.epoch, history_table.val_acc, label='accuracy'); validation_axis.set_title('validation'); validation_axis.legend()
plt.tight_layout(); plt.show()

### 11. Test results (touched once, with the best-validation weights)
Reloads the best checkpoint and predicts the test split. Overall accuracy is dominated by spirals (63% of the data), so read the per-class precision and recall in the report.
`smooth_cigar` is the smallest class here (238 test galaxies).

In [ ]:
model.load_weights(RUN_DIR/'best.weights.h5')
test_logits = model.predict(make_dataset(split_indices['test'], 512, False), verbose=0)
predicted_labels, true_labels = test_logits.argmax(1), labels[split_indices['test']]
print(f'TEST accuracy {(predicted_labels == true_labels).mean():.3f} | macro F1 {f1_score(true_labels, predicted_labels, average="macro"):.3f}\n')
all_class_ids = list(range(len(class_names)))
print(classification_report(true_labels, predicted_labels, labels=all_class_ids, target_names=class_names, digits=3, zero_division=0))

### 12. Confusion matrix
Each row is the true class and each column the predicted class, normalized so a row sums to 1: the diagonal is per-class recall, and the off-diagonal cells show which classes get mistaken for which.

In [ ]:
normalized_confusion = np.nan_to_num(confusion_matrix(true_labels, predicted_labels, labels=all_class_ids, normalize='true'))
figure, confusion_axis = plt.subplots(figsize=(6.5, 5.5)); confusion_axis.imshow(normalized_confusion, cmap='Blues', vmin=0, vmax=1)
confusion_axis.set_xticks(range(len(class_names)), class_names, rotation=45, ha='right'); confusion_axis.set_yticks(range(len(class_names)), class_names)
for true_class_id in range(len(class_names)):
    for predicted_class_id in range(len(class_names)):
        cell_value = normalized_confusion[true_class_id, predicted_class_id]
        confusion_axis.text(predicted_class_id, true_class_id, f'{cell_value:.2f}', ha='center', va='center', color='white' if cell_value > .5 else 'black', fontsize=8)
confusion_axis.set_xlabel('predicted'); confusion_axis.set_ylabel('true (row-normalised)'); plt.tight_layout(); plt.show()